# PAISA — OGDC frozen Random Forest training stage

This notebook builds the first reproducible model-training stage for PAISA.
It trains versioned Random Forest variants using technical PSX data only.

It does **not** perform portfolio simulation, live trading, news analysis,
sentiment analysis, fundamental analysis, reinforcement learning, or financial advice.

## 1. Environment detection

This cell detects Colab, Kaggle, or another Linux environment, prints system
information, and confirms that scikit-learn `RandomForestClassifier` uses CPU.

In [ ]:
from __future__ import annotations

import os
import platform
import sys
from pathlib import Path

def detect_runtime() -> str:
    if "COLAB_RELEASE_TAG" in os.environ:
        return "colab"
    if "KAGGLE_URL_BASE" in os.environ:
        return "kaggle"
    return "linux"

RUNTIME = detect_runtime()
if RUNTIME == "colab":
    WORKDIR = Path("/content")
    REPO_DIR = WORKDIR / "PAISA"
elif RUNTIME == "kaggle":
    WORKDIR = Path("/kaggle/working")
    REPO_DIR = WORKDIR / "PAISA"
else:
    WORKDIR = Path.cwd()
    REPO_DIR = WORKDIR if (WORKDIR / "pyproject.toml").exists() else WORKDIR / "PAISA"

print("Runtime:", RUNTIME)
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Machine:", platform.machine())
print("Processor:", platform.processor() or platform.machine())
try:
    pages = os.sysconf("SC_PHYS_PAGES")
    page_size = os.sysconf("SC_PAGE_SIZE")
    print("RAM GB:", round(pages * page_size / (1024**3), 2))
except Exception as exc:
    print("RAM GB: unavailable", exc)

try:
    import torch
    print("GPU available:", bool(torch.cuda.is_available()))
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception:
    print("GPU available: false or torch unavailable")

print("NOTE: scikit-learn RandomForestClassifier uses CPU in this run.")
print("Workdir:", WORKDIR)
print("Repo dir:", REPO_DIR)

## 2. Optional Google Drive mount

Run this cell only if you want to copy the final ZIP to Drive. It is intentionally optional.

In [ ]:
# Optional Colab-only cell. Leave commented for Kaggle.
# if RUNTIME == "colab":
#     from google.colab import drive
#     drive.mount("/content/drive")

## 3. Clone PAISA repository and install dependencies

The notebook clones the experiment branch if the repo is not already present.

In [ ]:
import subprocess

REPO_URL = "https://github.com/Arfa01/PAISA.git"
BRANCH = "feature/ogdc-rf-training-stage"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Repository already exists:", REPO_DIR)

%cd {REPO_DIR}
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

## 4. Deterministic seed setup and package versions

In [ ]:
import random
import numpy as np
import pandas as pd
import sklearn
import joblib
import matplotlib

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("matplotlib:", matplotlib.__version__)

## 5. Run the frozen OGDC training stage

This uses the existing PAISA pipeline with the real `psx-dps` provider, requests OGDC
from `2021-01-01`, and does not pass `--train` to the v0 pipeline. The new training
stage creates separate immutable artifacts under `artifacts/model_training/<run_id>/`.

In [ ]:
from pathlib import Path
from paisa.experiments.trainer import run_training_stage

summary = run_training_stage(
    symbol="OGDC",
    start="2021-01-01",
    provider="psx-dps",
    output_dir=Path("artifacts/model_training"),
    random_state=42,
)
summary

## 6. Show final summary files

In [ ]:
import json
from pathlib import Path

run_dir = Path(summary["artifact_directory"])
print("Run directory:", run_dir)
print("ZIP path:", summary["zip_path"])
print("ZIP size MB:", summary["zip_size_mb"])

final_summary = json.loads((run_dir / "reports" / "final_summary.json").read_text())
print(json.dumps(final_summary, indent=2)[:5000])

## 7. Inspect leaderboards and baselines

In [ ]:
validation_leaderboard = pd.read_csv(run_dir / "reports" / "validation_leaderboard.csv")
test_leaderboard = pd.read_csv(run_dir / "reports" / "test_leaderboard.csv")
baselines = pd.read_csv(run_dir / "reports" / "baselines.csv")

print("Validation leaderboard")
display(validation_leaderboard)

print("Final test leaderboard")
display(test_leaderboard)

print("Reference baselines")
display(baselines)

## 8. Display generated plots

In [ ]:
from IPython.display import Image, display

plot_names = [
    "price_and_splits.png",
    "class_distribution.png",
    "validation_model_comparison.png",
    "test_model_comparison.png",
    "rolling_5_day_hit_rate.png",
    "rolling_20_day_hit_rate.png",
    "cumulative_accuracy.png",
]

for name in plot_names:
    path = run_dir / "plots" / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

## 9. Final PASS/FAIL checklist

In [ ]:
checklist = json.loads((run_dir / "reports" / "final_pass_fail_checklist.json").read_text())
print(json.dumps(checklist, indent=2))
if not checklist["overall_pass"]:
    raise RuntimeError("Training stage did not pass final validation checklist.")
print("FINAL STATUS: PASS")

## 10. Optional Colab download cell

This cell is intentionally last and optional, so the notebook remains compatible with Kaggle.

In [ ]:
# Optional Colab-only download.
# if RUNTIME == "colab":
#     from google.colab import files
#     files.download(summary["zip_path"])